# Lab 4: Testing, Mocking, Logging, and Observability

### Instructions
Welcome to this week's lab! You will practice testing CLI tools with `pytest`, mocking external system dependencies using `monkeypatch`, configuring standard and structured logging, and verifying CLI packaging entry points.

* **Execution Environments:** All code in this notebook works across **macOS, Linux, and Windows**.
* **Submission:** Complete all code cells, short answer sections, and the bonus challenge directly inside this notebook file (`lab4_testing_logging.ipynb`).

## Part 1: Short Answer Questions

### Question 1.1: AAA Testing Pattern & Parametrization
Explain the "Arrange-Act-Assert" (AAA) pattern in unit testing. How does `@pytest.mark.parametrize` help enforce one behavior per test while avoiding code duplication?

*AAA Pattern: Arrange sets up test data, Act executes the code, and Assert verifies the outcome.
Parametrization: @pytest.mark.parametrize runs one test across multiple data sets without repeating test code*


### Question 1.2: `print()` vs. `logging` Stream Citizenship
Why is using `print()` considered problematic in automation tools, whereas `logging` provides proper stream citizenship? Explain where data vs. diagnostics should go.

*print() mixes logs and data together on stdout.

logging separates functional output (stdout) from diagnostic logs (stderr).*


## Part 2: Code Analysis & Bug Hunting

Read the following code snippets and answer the corresponding questions.

### Question 2.1: Logger Configuration at the Application Boundary
Analyze the following library module code:

```python
# network_scanner.py (Library Module)
import logging

logging.basicConfig(level=logging.DEBUG, filename="app.log")
log = logging.getLogger(__name__)

def scan_host(host: str):
    log.info(f"Scanning {host}")
    # ... scanning logic ...
```

1. **Bug Identification:** What rule of logging architecture does `logging.basicConfig(...)` violate when executed inside a library module?
2. **Impact Analysis:** How does this affect an importing CLI tool or application?
3. **Correct Pattern:** How should `logging` be instantiated inside a non-entrypoint module?

*Type your answer here:*

1. Violation: Libraries should not call logging.basicConfig().
2. Impact: Overrides global log settings for importing applications.
3. Correct Pattern: Use log = logging.getLogger(__name__).


### Question 2.2: Identifying Untestable Seams & Hardcoded Execution
Analyze the function below:

```python
import subprocess
import sys

def check_system_service(service_name: str):
    # Runs systemctl directly to check service state
    res = subprocess.run(["systemctl", "is-active", service_name], capture_output=True, text=True)
    if res.returncode == 0:
        print(f"SERVICE {service_name}: ACTIVE")
        return True
    else:
        print(f"SERVICE {service_name}: INACTIVE", file=sys.stderr)
        return False
```

1. Why is this function difficult to test in a CI/CD pipeline or on macOS/Windows developers' laptops?
2. How can `pytest`'s `monkeypatch` fixture be used to test both return paths without needing a Linux host running `systemd`?

*Type your answer here:*

1. Testability issue: Calling systemctl directly breaks on non-Linux OSs (Windows/macOS).
2. Mocking solution: Use monkeypatch in pytest to mock subprocess.run responses.


## Part 3: Code Completion Exercises

#### Note: All sections past here will be found in the downloaded .py files. This is due to jupyter notebooks not functioning properly with certain self calls.

Complete the missing sections marked with `# TODO` in the code blocks found in the files.

### Exercise 3.1: Parametrized Testing with `pytest` 
Go to the file "Lab4_Parametrization.py" and complete the #TODO sections.

Complete the test suite using `@pytest.mark.parametrize` to test a function that parses HTTP status codes into log classification buckets.

Expected output:
```
<directory>lab4Parametrization.py .....                                            [100%]

================================== 5 passed in 0.20s ===================================
```

### Exercise 3.2: Mocking Subprocess Calls with `monkeypatch` 
Go to the file "Lab4_MonkeyPatch.py" and complete the #TODO sections.

Complete the test function to mock `subprocess.run` using `monkeypatch` to simulate a process execution timeout without running actual system binaries.

Expected output:
```
================================= test session starts =================================
platform <platform> -- Python <version>, pytest-<version>, pluggy-<version>
rootdir: <directory>
collected 1 item

CS4310/TA/Lab4/lab3_32.py ERROR_TIMEOUT
.

================================== 1 passed in 0.03s ===================================
```

### Exercise 3.3: Creating a Custom Structured JSON Logger
Go to the file "Lab4_JSON.py" and complete the #TODO sections.

Complete the implementation of `JsonFormatter` so that log messages emit valid NDJSON strings containing `ts`, `level`, `logger`, and `msg` fields.

Expected output:
```
{"ts": <timestamp>, "level": "INFO", "logger": "service_audit", "msg": "Database connection established"}
```

### Exercise 3.4: Implement a Redacting Logging Filter

Write a custom `logging.Filter` class named `RedactingFilter` that automatically redacts sensitive keywords (e.g., tokens, passwords, API keys) from log messages before they hit stderr.

Go to the file "Lab4_LoggingFilter.py" and complete the #TODO sections.

**Requirements:**
1. Inherit from `logging.Filter`.
2. The constructor should accept a set or list of sensitive patterns/words to redact (e.g., `patterns={"secret_token_12345", "bearer_abc123"}`).
3. Implement `filter(self, record: logging.LogRecord) -> bool`:
   * Inspect `record.getMessage()` or `record.msg`.
   * Replace any occurrence of the sensitive tokens with `"***REDACTED***"`.
   * Modify `record.msg` and `record.args` in-place and return `True` so the record is emitted.
4. Verify cross-platform safety without modifying `os.environ`.

Expected output:
```
User authenticated with ***Redacted*** successfully.
Attempted login with password ***Redacted*** failed.
```